In [18]:
import pandas as pd
import numpy as np
from pathlib import Path

data_dir = Path("../Dataset")

aisles = pd.read_csv(data_dir / "aisles.csv")
departments = pd.read_csv(data_dir / "departments.csv")
orders = pd.read_csv(data_dir / "orders_cleaned.csv")
products = pd.read_csv(data_dir / "products.csv")

order_products_prior = pd.read_csv(
    data_dir / "order_products__prior.csv"
)

order_products_train = pd.read_csv(
    data_dir / "order_products__train.csv"
)

### Create stg_order_items

In [2]:
stg_order_items = pd.concat(
    [
        order_products_prior,
        order_products_train
    ],
    ignore_index=True
)

In [3]:
print("Prior:", order_products_prior.shape)
print("Train:", order_products_train.shape)
print("Combined:", stg_order_items.shape)

stg_order_items.head()

Prior: (32434489, 4)
Train: (1384617, 4)
Combined: (33819106, 4)


,order_id,product_id,add_to_cart_order,reordered
0,2,33120,1,1
1,2,28985,2,1
2,2,9327,3,0
3,2,45918,4,1
4,2,30035,5,0


### Create dim_product

The products, aisles and departments tables were combined to create dim_product. This provides one complete record for each product, including its product name, aisle and department. The dimension improves data readability, simplifies later analysis and supports product-category comparisons, reorder analysis and cross-selling identification. The original product_id was retained as the primary key because it is unique and stable within the dataset.

In [16]:
## Join products with aisles
dim_product = products.merge(
    aisles,
    on="aisle_id",
    how="left",
    validate="many_to_one"
)

In [15]:
## Then join with departments:
dim_product = dim_product.merge(
    departments,
    on="department_id",
    how="left",
    validate="many_to_one"
)

In [19]:
# Create dim_product from the original tables
dim_product = products.merge(
    aisles,
    on="aisle_id",
    how="left"
)

dim_product = dim_product.merge(
    departments,
    on="department_id",
    how="left"
)

# Arrange the columns
dim_product = dim_product[
    [
        "product_id",
        "product_name",
        "aisle_id",
        "aisle",
        "department_id",
        "department"
    ]
]

print("dim_product shape:", dim_product.shape)
dim_product.head()

dim_product shape: (49688, 6)


,product_id,product_name,aisle_id,aisle,department_id,department
0,1,Chocolate Sandwich Cookies,61,cookies cakes,19,snacks
1,2,All-Seasons Salt,104,spices seasonings,13,pantry
2,3,Robust Golden Unsweetened Oolong Tea,94,tea,7,beverages
3,4,Smart Ones Classic Favorites Mini Rigatoni Wit...,38,frozen meals,1,frozen
4,5,Green Chile Anytime Sauce,5,marinades meat preparation,13,pantry


In [46]:
## Arrange the columns:
dim_product = dim_product[
    [
        "product_id",
        "product_name",
        "aisle_id",
        "aisle",
        "department_id",
        "department"
    ]
]

print("dim_product shape:", dim_product.shape)
dim_product.head()

dim_product shape: (49688, 6)


,product_id,product_name,aisle_id,aisle,department_id,department
0,1,Chocolate Sandwich Cookies,61,cookies cakes,19,snacks
1,2,All-Seasons Salt,104,spices seasonings,13,pantry
2,3,Robust Golden Unsweetened Oolong Tea,94,tea,7,beverages
3,4,Smart Ones Classic Favorites Mini Rigatoni Wit...,38,frozen meals,1,frozen
4,5,Green Chile Anytime Sauce,5,marinades meat preparation,13,pantry


In [47]:
output_file = data_dir / "dim_product.csv"

dim_product.to_csv(
    output_file,
    index=False
)

print("Saved successfully:", output_file)

Saved successfully: ../Dataset/dim_product.csv


In [48]:
saved_dim_product = pd.read_csv(output_file)

print(saved_dim_product.shape)
display(saved_dim_product.head())

(49688, 6)


,product_id,product_name,aisle_id,aisle,department_id,department
0,1,Chocolate Sandwich Cookies,61,cookies cakes,19,snacks
1,2,All-Seasons Salt,104,spices seasonings,13,pantry
2,3,Robust Golden Unsweetened Oolong Tea,94,tea,7,beverages
3,4,Smart Ones Classic Favorites Mini Rigatoni Wit...,38,frozen meals,1,frozen
4,5,Green Chile Anytime Sauce,5,marinades meat preparation,13,pantry


# Order Level Feature

### Create day_part

In [20]:
orders_fe = orders.copy()

orders_fe["day_part"] = pd.cut(
    orders_fe["order_hour_of_day"],
    bins=[-1, 5, 11, 17, 23],
    labels=[
        "Night",
        "Morning",
        "Afternoon",
        "Evening"
    ]
)

orders_fe[
    ["order_id", "order_hour_of_day", "day_part"]
].head()

,order_id,order_hour_of_day,day_part
0,2539329,8,Morning
1,2398795,7,Morning
2,473747,12,Afternoon
3,2254736,7,Morning
4,431534,15,Afternoon


### Create item_count


In [21]:
stg_order_items["item_count"] = 1

stg_order_items.head()

,order_id,product_id,add_to_cart_order,reordered,item_count
0,2,33120,1,1,1
1,2,28985,2,1,1
2,2,9327,3,0,1
3,2,45918,4,1,1
4,2,30035,5,0,1


### Create order-level features

In [22]:
order_features = (
    stg_order_items
    .groupby("order_id")
    .agg(
        basket_size=("item_count", "sum"),
        reordered_item_count=("reordered", "sum")
    )
    .reset_index()
)

order_features.head()

,order_id,basket_size,reordered_item_count
0,1,8,4
1,2,9,6
2,3,8,8
3,4,13,12
4,5,26,21


### Create reorder_rate

In [23]:
order_features["reorder_rate"] = (
    order_features["reordered_item_count"] /
    order_features["basket_size"]
)

order_features.head()

,order_id,basket_size,reordered_item_count,reorder_rate
0,1,8,4,0.500000
1,2,9,6,0.666667
2,3,8,8,1.000000
3,4,13,12,0.923077
4,5,26,21,0.807692


### Combine with order information

In [24]:
orders_features = orders_fe.merge(
    order_features,
    on="order_id",
    how="inner",
    validate="one_to_one"
)

orders_features[
    [
        "order_id",
        "user_id",
        "order_hour_of_day",
        "day_part",
        "basket_size",
        "reordered_item_count",
        "reorder_rate"
    ]
].head()

,order_id,user_id,order_hour_of_day,day_part,basket_size,reordered_item_count,reorder_rate
0,2539329,1,8,Morning,5,0,0.000
1,2398795,1,7,Morning,6,3,0.500
2,473747,1,12,Afternoon,5,3,0.600
3,2254736,1,7,Morning,5,5,1.000
4,431534,1,15,Afternoon,8,5,0.625


Step 1: Create day_part

What it does: Groups the 24 hours into Night, Morning, Afternoon and Evening.

Rationale: It is easier to compare four shopping periods than 24 separate hours. This helps identify when customers prefer to shop.

Step 2: Create item_count

What it does: Gives every order-product row a value of 1.

Rationale: Each row represents one product added to an order. Adding the 1s together tells us how many products are in each order.

Example:

1 + 1 + 1 + 1 = 4 products
Step 3: Create basket_size

What it does: Counts the number of products in each order.

Rationale: It shows whether the customer made a small or large purchase. Large baskets may provide more cross-selling opportunities.

Step 4: Create reordered_item_count

What it does: Counts how many products in an order were purchased before.

Rationale: It helps identify repeat-purchase behaviour. A high count suggests that the customer regularly returns to familiar products.

Step 5: Create reorder_rate

Calculation:

Reordered items ÷ Basket size

Rationale: The count alone may be misleading because basket sizes differ. The rate makes orders easier to compare.

Example:

8 reordered products ÷ 10 products = 80% reorder rate

This means most products in the basket were repeat purchases.

Step 6: Combine with order information

What it does: Adds the new basket features to information such as customer, hour and day.

Rationale: This creates one useful order-level table. We can then study questions such as:

When do customers place large orders?
Which customers have high reorder rates?
Does shopping behaviour change by time of day?
Which baskets offer cross-selling opportunities?


# customer-level features
Customer-level features were created by aggregating order and product records by user_id. These features describe customer activity, purchase frequency, basket size, product variety and reorder behaviour. The resulting dim_user table provides one record per customer and supports customer segmentation and cross-selling analysis.

create the five customer-level features and combine them into dim_user.

### 1. Total orders and average days between orders
Calculation: Count the number of orders made by each customer.

Rationale: Shows how active or loyal a customer is.

Example:

5 orders → occasional customer
80 orders → frequent customer

In [26]:
customer_order_features = (
    orders
    .groupby("user_id")
    .agg(
        total_orders=("order_id", "nunique"),
        average_days_between_orders=(
            "days_since_prior_order",
            "mean"
        )
    )
    .reset_index()
)

customer_order_features.head()

,user_id,total_orders,average_days_between_orders
0,1,11,19.000000
1,2,15,16.285714
2,3,13,12.000000
3,4,6,17.000000
4,5,5,11.500000


In [15]:
user_1 = orders[
    orders["user_id"] == 1
]

display(
    user_1[
        [
            "user_id",
            "order_id",
            "order_number",
            "days_since_prior_order"
        ]
    ]
)

,user_id,order_id,order_number,days_since_prior_order
0,1,2539329,1,NaN
1,1,2398795,2,15.0
2,1,473747,3,21.0
3,1,2254736,4,29.0
4,1,431534,5,28.0
5,1,3367565,6,19.0
6,1,550135,7,20.0
7,1,3108588,8,14.0
8,1,2295261,9,0.0
9,1,2550362,10,30.0


### 2. Average basket size and reorder totals

Calculation: Average number of products in the customer’s orders.

Rationale: Shows the customer’s typical purchasing volume.

Example:

Average of 3 products → small-basket shopper
Average of 25 products → large-basket shopper

Large-basket customers may offer more cross-selling opportunities.

In [16]:
# Create item count
stg_order_items["item_count"] = 1

# Create order-level features
order_features = (
    stg_order_items
    .groupby("order_id")
    .agg(
        basket_size=("item_count", "sum"),
        reordered_item_count=("reordered", "sum")
    )
    .reset_index()
)

# Create reorder rate
order_features["reorder_rate"] = (
    order_features["reordered_item_count"] /
    order_features["basket_size"]
)

# Add the features to the orders table
orders_features = orders.merge(
    order_features,
    on="order_id",
    how="inner",
    validate="one_to_one"
)

orders_features[
    [
        "order_id",
        "user_id",
        "basket_size",
        "reordered_item_count",
        "reorder_rate"
    ]
].head()

,order_id,user_id,basket_size,reordered_item_count,reorder_rate
0,2539329,1,5,0,0.000
1,2398795,1,6,3,0.500
2,473747,1,5,3,0.600
3,2254736,1,5,5,1.000
4,431534,1,8,5,0.625


### 3. Average-days_between_orders
Calculation: Average number of days between the customer’s orders.
Rationale: Shows how frequently the customer returns.

Example:

5 days → shops frequently
25 days → shops less frequently

The first order’s missing value is automatically excluded because it has no previous order.
Calculation: Average number of days between the customer’s orders.




In [31]:
average_days_between_orders=(
    "days_since_prior_order",
    "mean"
)

display(customer_order_features.head())

,user_id,total_orders,average_days_between_orders
0,1,11,19.000000
1,2,15,16.285714
2,3,13,12.000000
3,4,6,17.000000
4,5,5,11.500000


### 4. unique_products_purchased (create products feature)

Calculation: Count the number of different products purchased by each customer.

Rationale: Shows the variety of the customer’s product choices.

Example:

10 unique products → buys a small, regular selection
200 unique products → explores many products


In [32]:
customer_product_features = (
    stg_order_items
    .groupby("user_id")
    .agg(
        unique_products_purchased=(
            "product_id",
            "nunique"
        )
    )
    .reset_index()
)

customer_product_features.head()

,user_id,unique_products_purchased
0,1,19
1,2,121
2,3,33
3,4,17
4,5,28


In [37]:
order_to_user = orders.set_index(
    "order_id"
)["user_id"]

stg_order_items["user_id"] = (
    stg_order_items["order_id"]
    .map(order_to_user)
)

customer_product_features = (
    stg_order_items
    .groupby("user_id")
    .agg(
        unique_products_purchased=(
            "product_id",
            "nunique"
        )
    )
    .reset_index()
)

display(customer_product_features.head())

,user_id,unique_products_purchased
0,1,19
1,2,121
2,3,33
3,4,17
4,5,28


### 5. Create Basket Feature
Customer basket features were created by grouping order-level records by user_id. These features summarise each customer’s typical basket size, total purchasing volume and repeat-purchase behaviour. They support customer segmentation and help identify customer loyalty and cross-selling opportunities.

In [34]:
customer_basket_features = (
    orders_features
    .groupby("user_id")
    .agg(
        average_basket_size=("basket_size", "mean"),
        total_items=("basket_size", "sum"),
        total_reordered_items=(
            "reordered_item_count",
            "sum"
        )
    )
    .reset_index()
)

customer_basket_features["customer_reorder_rate"] = (
    customer_basket_features["total_reordered_items"] /
    customer_basket_features["total_items"]
)

display(customer_basket_features.head())

,user_id,average_basket_size,total_items,total_reordered_items,customer_reorder_rate
0,1,6.363636,70,51,0.728571
1,2,15.066667,226,105,0.464602
2,3,7.333333,88,55,0.625000
3,4,3.600000,18,1,0.055556
4,5,9.200000,46,18,0.391304


### Customer reorder rate

customer_reorder_rate is stored as a decimal because decimals are easier to use in calculations and modelling.

0.65

The percentage feature converts it into a more understandable business format:

0.65 × 100 = 65%

This helps users quickly interpret customer loyalty:

High percentage → customer frequently repurchases familiar products.
Low percentage → customer explores more new products.

Displaying total_items and total_reordered_items also makes the calculation transparent and easier to verify.

Report wording

Customer reorder percentage was derived by multiplying the customer reorder rate by 100. The percentage format improves readability and supports easier comparison of repeat-purchase behaviour across customers. The original decimal rate was retained for calculations, while the percentage was used for interpretation and dashboard presentation.

In [39]:
customer_basket_features[
    "customer_reorder_percentage"
] = (
    customer_basket_features["customer_reorder_rate"]
    * 100
).round(2)

display(
    customer_basket_features[
        [
            "user_id",
            "average_basket_size",
            "total_items",
            "total_reordered_items",
            "customer_reorder_rate",
            "customer_reorder_percentage"
        ]
    ].head(10)
)

,user_id,average_basket_size,total_items,total_reordered_items,customer_reorder_rate,customer_reorder_percentage
0,1,6.363636,70,51,0.728571,72.86
1,2,15.066667,226,105,0.464602,46.46
2,3,7.333333,88,55,0.625000,62.50
3,4,3.600000,18,1,0.055556,5.56
4,5,9.200000,46,18,0.391304,39.13
5,6,4.666667,14,2,0.142857,14.29
6,7,10.238095,215,146,0.679070,67.91
7,8,16.750000,67,17,0.253731,25.37
8,9,24.500000,98,40,0.408163,40.82
9,10,24.500000,147,49,0.333333,33.33


In [40]:
user_1 = orders_features[
    orders_features["user_id"] == 1
]

total_items = user_1["basket_size"].sum()
total_reordered = user_1["reordered_item_count"].sum()

print("Total items:", total_items)
print("Reordered items:", total_reordered)
print("Reorder rate:", total_reordered / total_items)

Total items: 70
Reordered items: 51
Reorder rate: 0.7285714285714285


### Why create dim_user?

The original dataset does not provide a customer table. It only provides user_id inside orders.csv.

We therefore group the orders and purchases by user_id to create one row describing each customer.

In [41]:

dim_user = (
    customer_order_features
    .merge(
        customer_basket_features,
        on="user_id",
        how="left",
        validate="one_to_one"
    )
    .merge(
        customer_product_features,
        on="user_id",
        how="left",
        validate="one_to_one"
    )
)

display(dim_user.head())

,user_id,total_orders,average_days_between_orders,average_basket_size,total_items,total_reordered_items,customer_reorder_rate,customer_reorder_percentage,unique_products_purchased
0,1,11,19.000000,6.363636,70,51,0.728571,72.86,19
1,2,15,16.285714,15.066667,226,105,0.464602,46.46,121
2,3,13,12.000000,7.333333,88,55,0.625000,62.50,33
3,4,6,17.000000,3.600000,18,1,0.055556,5.56,17
4,5,5,11.500000,9.200000,46,18,0.391304,39.13,28


In [42]:
dim_user = (
    customer_order_features
    .merge(
        customer_basket_features,
        on="user_id",
        how="left",
        validate="one_to_one"
    )
    .merge(
        customer_product_features,
        on="user_id",
        how="left",
        validate="one_to_one"
    )
)

In [43]:
dim_user = dim_user[
    [
        "user_id",
        "total_orders",
        "average_days_between_orders",
        "average_basket_size",
        "total_items",
        "total_reordered_items",
        "unique_products_purchased",
        "customer_reorder_rate",
        "customer_reorder_percentage"
    ]
]

In [44]:
output_file = data_dir / "dim_user_features.csv"

dim_user.to_csv(
    output_file,
    index=False
)

print("Saved successfully:", output_file)

Saved successfully: ../Dataset/dim_user_features.csv


In [45]:
saved_dim_user = pd.read_csv(output_file)

print(saved_dim_user.shape)
display(saved_dim_user.head())

(206209, 9)


,user_id,total_orders,average_days_between_orders,average_basket_size,total_items,total_reordered_items,unique_products_purchased,customer_reorder_rate,customer_reorder_percentage
0,1,11,19.000000,6.363636,70,51,19,0.728571,72.86
1,2,15,16.285714,15.066667,226,105,121,0.464602,46.46
2,3,13,12.000000,7.333333,88,55,33,0.625000,62.50
3,4,6,17.000000,3.600000,18,1,17,0.055556,5.56
4,5,5,11.500000,9.200000,46,18,28,0.391304,39.13


# Dim_order_slot
dim_order_slot stores each day-and-hour combination once. The engineered day_part field groups individual hours into Night, Morning, Afternoon and Evening, making ordering patterns easier to analyse and visualise. The slot_key provides a unique primary key for connecting the dimension to fact_order

In [49]:
# Select day and hour columns
dim_order_slot = (
    orders[["order_dow", "order_hour_of_day"]]
    .drop_duplicates()
    .sort_values(["order_dow", "order_hour_of_day"])
    .reset_index(drop=True)
)

# Create day_part feature
dim_order_slot["day_part"] = pd.cut(
    dim_order_slot["order_hour_of_day"],
    bins=[-1, 5, 11, 17, 23],
    labels=["Night", "Morning", "Afternoon", "Evening"]
)

# Create primary key
dim_order_slot.insert(
    0,
    "slot_key",
    range(1, len(dim_order_slot) + 1)
)

display(dim_order_slot.head(20))

,slot_key,order_dow,order_hour_of_day,day_part
0,1,0,0,Night
1,2,0,1,Night
2,3,0,2,Night
3,4,0,3,Night
4,5,0,4,Night
5,6,0,5,Night
6,7,0,6,Morning
7,8,0,7,Morning
8,9,0,8,Morning
9,10,0,9,Morning


In [50]:
print("Shape:", dim_order_slot.shape)
print("\nMissing values:")
print(dim_order_slot.isna().sum())

print(
    "\nDuplicate day-hour combinations:",
    dim_order_slot.duplicated(
        subset=["order_dow", "order_hour_of_day"]
    ).sum()
)

Shape: (168, 4)

Missing values:
slot_key             0
order_dow            0
order_hour_of_day    0
day_part             0
dtype: int64

Duplicate day-hour combinations: 0


In [66]:
dim_order_slot.to_csv(
    "../Dataset/dim_order_slot.csv",
    index=False
)

print("Saved successfully: ../Dataset/dim_order_slot.csv")

Saved successfully: ../Dataset/dim_order_slot.csv


# Fact order
#### fact_order stores one row for each order. It connects customers to ordering time slots and contains engineered basket measures. Keeping these measures at the order level prevents basket size and reorder rate from being repeated for every product.

#### Step 1: Calculate prior-order features

In [52]:
prior_order_features = (
    order_products_prior
    .groupby("order_id")
    .agg(
        basket_size=("product_id", "count"),
        reordered_item_count=("reordered", "sum")
    )
    .reset_index()
)

#### Step 2: Calculate train-order features

In [53]:
train_order_features = (
    order_products_train
    .groupby("order_id")
    .agg(
        basket_size=("product_id", "count"),
        reordered_item_count=("reordered", "sum")
    )
    .reset_index()
)

#### Step 3: Combine the order features

In [54]:
order_features = pd.concat(
    [
        prior_order_features,
        train_order_features
    ],
    ignore_index=True
)

#### Step 4: Calculate reorder rate

In [55]:
order_features["reorder_rate"] = (
    order_features["reordered_item_count"]
    / order_features["basket_size"]
)

In [56]:
display(order_features.head())

,order_id,basket_size,reordered_item_count,reorder_rate
0,2,9,6,0.666667
1,3,8,8,1.000000
2,4,13,12,0.923077
3,5,26,21,0.807692
4,6,3,0,0.000000


#### Step 5: Add slot_key to orders

In [57]:
fact_order = orders.merge(
    dim_order_slot[
        [
            "slot_key",
            "order_dow",
            "order_hour_of_day"
        ]
    ],
    on=[
        "order_dow",
        "order_hour_of_day"
    ],
    how="left"
)

#### Step 6: Add the engineered order features

In [58]:
fact_order = fact_order.merge(
    order_features,
    on="order_id",
    how="left"
)

#### Step 7: Select the final columns

In [60]:
fact_order = fact_order[
    [
        "order_id",
        "user_id",
        "slot_key",
        "eval_set",
        "order_number",
        "days_since_prior_order",
        "basket_size",
        "reordered_item_count",
        "reorder_rate"
    ]
]

In [61]:
print("fact_order shape:", fact_order.shape)

display(fact_order.head())

fact_order shape: (3421083, 9)


,order_id,user_id,slot_key,eval_set,order_number,days_since_prior_order,basket_size,reordered_item_count,reorder_rate
0,2539329,1,57,prior,1,NaN,5.0,0.0,0.000
1,2398795,1,80,prior,2,15.0,6.0,3.0,0.500
2,473747,1,85,prior,3,21.0,5.0,3.0,0.600
3,2254736,1,104,prior,4,29.0,5.0,5.0,1.000
4,431534,1,112,prior,5,28.0,8.0,5.0,0.625


#### Step 8: Validate fact_order

In [62]:
print("Duplicate order_id:")
print(fact_order["order_id"].duplicated().sum())

print("\nMissing values:")
print(fact_order.isna().sum())

Duplicate order_id:
0

Missing values:
order_id                       0
user_id                        0
slot_key                       0
eval_set                       0
order_number                   0
days_since_prior_order    206209
basket_size                75000
reordered_item_count       75000
reorder_rate               75000
dtype: int64


### This result is correct.

Duplicate order_id = 0: every order is unique.
days_since_prior_order = 206,209 missing: every customer’s first order has no previous order.
basket_size = 75,000 missing: the 75,000 test orders have no released product details.
reordered_item_count = 75,000 missing: cannot calculate without test-order products.
reorder_rate = 75,000 missing: cannot calculate without basket information.

In [75]:
# Missing days should only belong to first orders
print(
    fact_order.loc[
        fact_order["days_since_prior_order"].isna(),
        "order_number"
    ].value_counts()
)

1    206209
Name: order_number, dtype: int64


In [76]:
# Missing basket features should only belong to test orders
print(
    fact_order.loc[
        fact_order["basket_size"].isna(),
        "eval_set"
    ].value_counts()
)

test    75000
Name: eval_set, dtype: int64


In [77]:
fact_order.to_csv(
    "../Dataset/fact_order.csv",
    index=False,
    na_rep="NaN"
)

print("fact_order.csv saved with missing values as NaN.")

fact_order.csv saved with missing values as NaN.


In [78]:
fact_order = pd.read_csv(
    "../Dataset/fact_order.csv"
)

In [73]:
print(fact_order.isna().sum())

order_id                       0
user_id                        0
slot_key                       0
eval_set                       0
order_number                   0
days_since_prior_order    206209
basket_size                75000
reordered_item_count       75000
reorder_rate               75000
dtype: int64


# Create Fact Order Items
The fact_order_item table was created to store product-level transaction details for each order. It resolves the many-to-many relationship between orders and products and supports product-demand, reorder and market-basket analysis. Separating product-level records from order-level records prevents double-counting and maintains a clear table grain.

In [67]:
fact_item_columns = [
    "order_id",
    "product_id",
    "add_to_cart_order",
    "reordered"
]

output_path = "../Dataset/fact_order_item.csv"

# Save prior records first
order_products_prior[fact_item_columns].to_csv(
    output_path,
    index=False
)

# Append train records
order_products_train[fact_item_columns].to_csv(
    output_path,
    mode="a",
    header=False,
    index=False
)

print("Saved successfully:", output_path)
print(
    "Total rows:",
    len(order_products_prior) + len(order_products_train)
)

Saved successfully: ../Dataset/fact_order_item.csv
Total rows: 33819106


In [68]:
from zipfile import ZipFile, ZIP_DEFLATED

csv_path = "../Dataset/fact_order_item.csv"
zip_path = "../Dataset/fact_order_item.zip"

with ZipFile(
    zip_path,
    mode="w",
    compression=ZIP_DEFLATED
) as zip_file:
    zip_file.write(
        csv_path,
        arcname="fact_order_item.csv"
    )

print("ZIP file saved successfully:")
print(zip_path)

ZIP file saved successfully:
../Dataset/fact_order_item.zip


# Test the transformed tables

In [81]:
print(
    "dim_user duplicate user_id:",
    dim_user["user_id"].duplicated().sum()
)

print(
    "dim_product duplicate product_id:",
    dim_product["product_id"].duplicated().sum()
)

print(
    "dim_order_slot duplicate slot_key:",
    dim_order_slot["slot_key"].duplicated().sum()
)

print(
    "fact_order duplicate order_id:",
    fact_order["order_id"].duplicated().sum()
)

dim_user duplicate user_id: 0
dim_product duplicate product_id: 0
dim_order_slot duplicate slot_key: 0
fact_order duplicate order_id: 0


In [82]:
import pandas as pd

dim_user = pd.read_csv(
    "../Dataset/dim_user_features.csv"
)

dim_product = pd.read_csv(
    "../Dataset/dim_product.csv"
)

dim_order_slot = pd.read_csv(
    "../Dataset/dim_order_slot.csv"
)

fact_order = pd.read_csv(
    "../Dataset/fact_order.csv"
)

print("All four tables loaded.")

All four tables loaded.


In [83]:
print(
    "dim_user duplicate user_id:",
    dim_user["user_id"].duplicated().sum()
)

print(
    "dim_product duplicate product_id:",
    dim_product["product_id"].duplicated().sum()
)

print(
    "dim_order_slot duplicate slot_key:",
    dim_order_slot["slot_key"].duplicated().sum()
)

print(
    "fact_order duplicate order_id:",
    fact_order["order_id"].duplicated().sum()
)

dim_user duplicate user_id: 0
dim_product duplicate product_id: 0
dim_order_slot duplicate slot_key: 0
fact_order duplicate order_id: 0


In [84]:
print(
    "All fact_order users matched:",
    fact_order["user_id"].isin(
        dim_user["user_id"]
    ).all()
)

print(
    "All fact_order slots matched:",
    fact_order["slot_key"].isin(
        dim_order_slot["slot_key"]
    ).all()
)

All fact_order users matched: True
All fact_order slots matched: True


In [85]:
print(
    "Valid reorder rates:",
    fact_order["reorder_rate"]
    .dropna()
    .between(0, 1)
    .all()
)

observed_orders = fact_order[
    fact_order["eval_set"].isin(["prior", "train"])
]

print(
    "Valid basket sizes:",
    observed_orders["basket_size"]
    .gt(0)
    .all()
)

Valid reorder rates: True
Valid basket sizes: True


In [87]:
print(
    "Previous-order days missing only for first orders:",
    (
        fact_order["days_since_prior_order"].isna()
        == fact_order["order_number"].eq(1)
    ).all()
)
test_orders = fact_order[
    fact_order["eval_set"] == "test"
]

print(
    "Test-order basket features are missing:",
    test_orders[
        [
            "basket_size",
            "reordered_item_count",
            "reorder_rate"
        ]
    ].isna().all().all()
)



Previous-order days missing only for first orders: True
Test-order basket features are missing: True


In [88]:
print("dim_user rows:", len(dim_user))
print("dim_product rows:", len(dim_product))
print("dim_order_slot rows:", len(dim_order_slot))
print("fact_order rows:", len(fact_order))

dim_user rows: 206209
dim_product rows: 49688
dim_order_slot rows: 168
fact_order rows: 3421083


## We create separate tables because each table represents a different type of information.

#### Table	One row represents
#### dim_user	One customer
#### dim_product	One product
#### dim_order_slot	One day-and-hour combination
#### fact_order	One complete order
#### fact_order_item	One product inside an order

#### Think of a supermarket system:

#### dim_user = customer list
#### dim_product = product catalogue
#### dim_order_slot = ordering-time reference
#### fact_order = receipt summary
#### fact_order_item = products printed on each receipt